# FloodSegmentationAgent-HighPerf (v2026)

Notebook ini mengimplementasikan arsitektur performa tinggi untuk segmentasi banjir:

- Encoder: EfficientNet-B0 (ImageNet pretrained)
- Decoder: Attention U-Net dengan CBAM
- Skip connection: U-Net++ (nested dense skip)
- Bottleneck: Multi-Head Self-Attention
- Loss: Dice + BCE + Boundary loss
- Optimizer: AdamW
- Scheduler: Warmup + Cosine Annealing
- Post-processing: TTA + DenseCRF (opsional)

Catatan: target IoU 0.92-0.97 adalah target eksperimen, hasil aktual bergantung pada split data, kualitas label, dan hardware/runtime.


In [1]:
!nvidia-smi

Mon Mar 30 07:27:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [50]:
# Install dependencies
%pip install -q --upgrade torch torchvision timm albumentations opencv-python matplotlib pandas numpy scikit-learn kagglehub pillow tqdm

# Optional DenseCRF (boleh gagal install di beberapa runtime)
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydensecrf"], check=False)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'pydensecrf'], returncode=1)

In [51]:
import os
import cv2
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import kagglehub

try:
    import pydensecrf.densecrf as dcrf
    from pydensecrf.utils import unary_from_softmax
    HAS_DCRF = True
except Exception:
    HAS_DCRF = False

warnings.filterwarnings("ignore")

CONFIG = {
    "SEED": 42,
    "BATCH_SIZE": 16,  # effective batch size target
    "MICRO_BATCH_SIZE": 2,  # fallback aman untuk GPU terbatas
    "EPOCHS": 150,
    "FREEZE_EPOCHS": 30,
    "LR": 1e-4,
    "LR_UNFREEZE": 1e-5,
    "WEIGHT_DECAY": 1e-5,
    "IMAGE_HEIGHT": 512,
    "IMAGE_WIDTH": 512,
    "VAL_SIZE": 0.2,
    "NUM_WORKERS": 8,
    "CUTMIX_PROB": 0.3,
    "CUTMIX_ALPHA": 1.0,
    "WARMUP_EPOCHS": 5,
    "CHECKPOINT_PATH": "floodseg_highperf_best.pth",
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
}

print(f"Device: {CONFIG['DEVICE']}")
print(f"DenseCRF available: {HAS_DCRF}")

Device: cuda
DenseCRF available: False


In [52]:
# Optional authentication untuk akses dataset Kaggle
try:
    kagglehub.login()
except Exception as e:
    print("Login Kaggle opsional, lanjut jika dataset publik/cache tersedia.")
    print("Detail:", str(e)[:200])

In [53]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def clean_name(value):
    return str(value).strip().replace("\\", "/")


def resolve_existing_path(base_dir, root_dir, file_name):
    if os.path.isabs(file_name) and os.path.exists(file_name):
        return file_name

    candidates = [
        os.path.join(base_dir, file_name),
        os.path.join(root_dir, file_name),
        os.path.join(base_dir, os.path.basename(file_name)),
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]


def read_image_rgb(img_path):
    image = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if image is not None:
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    try:
        with Image.open(img_path) as img:
            return np.array(img.convert("RGB"))
    except Exception:
        return None


def read_mask_gray(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is not None:
        return mask

    try:
        with Image.open(mask_path) as m:
            return np.array(m.convert("L"))
    except Exception:
        return None


set_seed(CONFIG["SEED"])

In [54]:
# Dataset source sama seperti notebook UNET
dataset_root = kagglehub.dataset_download("faizalkarim/flood-area-segmentation")
print(f"Dataset root: {dataset_root}")

csv_path = os.path.join(dataset_root, "metadata.csv")
image_dir = os.path.join(dataset_root, "Image")
mask_dir = os.path.join(dataset_root, "Mask")
root_dir = os.path.dirname(image_dir)

metadata = pd.read_csv(csv_path)
valid_samples = []
missing_count = 0
decode_fail_count = 0

for i in range(len(metadata)):
    img_name = clean_name(metadata.iloc[i, 0])
    mask_name = clean_name(metadata.iloc[i, 1])

    img_path = resolve_existing_path(image_dir, root_dir, img_name)
    mask_path = resolve_existing_path(mask_dir, root_dir, mask_name)

    if not (os.path.isfile(img_path) and os.path.isfile(mask_path)):
        missing_count += 1
        continue

    image = read_image_rgb(img_path)
    mask = read_mask_gray(mask_path)
    if image is None or mask is None:
        decode_fail_count += 1
        continue

    valid_samples.append((img_path, mask_path))

if len(valid_samples) == 0:
    raise RuntimeError("Tidak ada sample valid. Cek path dataset/metadata.")

train_samples, val_samples = train_test_split(
    valid_samples,
    test_size=CONFIG["VAL_SIZE"],
    random_state=CONFIG["SEED"],
    shuffle=True,
 )

print(f"Total metadata: {len(metadata)}")
print(f"Valid samples: {len(valid_samples)}")
print(f"Missing file skipped: {missing_count}")
print(f"Decode fail skipped: {decode_fail_count}")
print(f"Train: {len(train_samples)} | Val: {len(val_samples)}")

Using Colab cache for faster access to the 'flood-area-segmentation' dataset.
Dataset root: /kaggle/input/flood-area-segmentation
Total metadata: 290
Valid samples: 290
Missing file skipped: 0
Decode fail skipped: 0
Train: 232 | Val: 58


In [55]:
class FloodDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = read_image_rgb(img_path)
        mask = read_mask_gray(mask_path)

        if image is None:
            raise FileNotFoundError(f"Gagal membaca image pada idx={idx}: {img_path}")
        if mask is None:
            raise FileNotFoundError(f"Gagal membaca mask pada idx={idx}: {mask_path}")

        if image.ndim == 2:
            image = np.stack([image] * 3, axis=-1)

        mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        mask = mask.unsqueeze(0).float()
        return image, mask


train_transform = A.Compose(
    [
        A.Resize(CONFIG["IMAGE_HEIGHT"], CONFIG["IMAGE_WIDTH"]),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.ShiftScaleRotate(
            shift_limit=0.10,
            scale_limit=0.20,
            rotate_limit=30,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.6,
        ),
        A.RandomBrightnessContrast(p=0.5),
        A.RandomGamma(p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ],
    is_check_shapes=False,
 )

val_transform = A.Compose(
    [
        A.Resize(CONFIG["IMAGE_HEIGHT"], CONFIG["IMAGE_WIDTH"]),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ],
    is_check_shapes=False,
 )

train_dataset = FloodDataset(train_samples, transform=train_transform)
val_dataset = FloodDataset(val_samples, transform=val_transform)

num_workers = min(CONFIG["NUM_WORKERS"], os.cpu_count() or 2)
pin_memory = CONFIG["DEVICE"] == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["MICRO_BATCH_SIZE"],
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=True,
 )
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["MICRO_BATCH_SIZE"],
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
 )

effective_batch = CONFIG["BATCH_SIZE"]
micro_batch = CONFIG["MICRO_BATCH_SIZE"]
accum_steps = max(1, math.ceil(effective_batch / micro_batch))
print(
    f"num_workers={num_workers} | train_batches={len(train_loader)} | val_batches={len(val_loader)} | "
    f"micro_batch={micro_batch} | effective_batch~{micro_batch * accum_steps} (accum_steps={accum_steps})"
)

num_workers=2 | train_batches=116 | val_batches=29 | micro_batch=2 | effective_batch~16 (accum_steps=8)


In [56]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x))
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(out)
        return self.sigmoid(out)


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction=reduction)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x


class AttentionGate(nn.Module):
    def __init__(self, skip_channels, gate_channels, inter_channels):
        super().__init__()
        self.theta = nn.Conv2d(skip_channels, inter_channels, kernel_size=1, bias=False)
        self.phi = nn.Conv2d(gate_channels, inter_channels, kernel_size=1, bias=False)
        self.psi = nn.Conv2d(inter_channels, 1, kernel_size=1, bias=True)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, skip, gate):
        if gate.shape[-2:] != skip.shape[-2:]:
            gate = F.interpolate(gate, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        attn = self.relu(self.theta(skip) + self.phi(gate))
        attn = self.sigmoid(self.psi(attn))
        return skip * attn


class MHSA2DBottleneck(nn.Module):
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(channels)
        self.ffn = nn.Sequential(
            nn.Linear(channels, channels * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(channels * 4, channels),
        )

    def forward(self, x):
        b, c, h, w = x.shape
        seq = x.flatten(2).transpose(1, 2)  # [B, N, C]

        attn_in = self.norm1(seq)
        attn_out, _ = self.attn(attn_in, attn_in, attn_in, need_weights=False)
        seq = seq + attn_out

        ffn_in = self.norm2(seq)
        seq = seq + self.ffn(ffn_in)

        return seq.transpose(1, 2).reshape(b, c, h, w)


class HighPerfFloodUNetPP(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(
            "efficientnet_b0",
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3, 4),
        )

        enc_ch = self.encoder.feature_info.channels()  # expected: [16,24,40,112,320]
        dec_ch = [32, 64, 128, 256, 320]

        self.proj0 = nn.Conv2d(enc_ch[0], dec_ch[0], kernel_size=1)
        self.proj1 = nn.Conv2d(enc_ch[1], dec_ch[1], kernel_size=1)
        self.proj2 = nn.Conv2d(enc_ch[2], dec_ch[2], kernel_size=1)
        self.proj3 = nn.Conv2d(enc_ch[3], dec_ch[3], kernel_size=1)
        self.proj4 = nn.Conv2d(enc_ch[4], dec_ch[4], kernel_size=1)

        self.cbam0 = CBAM(dec_ch[0])
        self.cbam1 = CBAM(dec_ch[1])
        self.cbam2 = CBAM(dec_ch[2])
        self.cbam3 = CBAM(dec_ch[3])
        self.cbam4 = CBAM(dec_ch[4])

        self.ag0 = AttentionGate(dec_ch[0], dec_ch[1], inter_channels=16)
        self.ag1 = AttentionGate(dec_ch[1], dec_ch[2], inter_channels=32)
        self.ag2 = AttentionGate(dec_ch[2], dec_ch[3], inter_channels=64)
        self.ag3 = AttentionGate(dec_ch[3], dec_ch[4], inter_channels=128)

        self.mhsa = MHSA2DBottleneck(dec_ch[4], num_heads=8, dropout=0.1)

        self.conv0_1 = ConvBlock(dec_ch[0] + dec_ch[1], dec_ch[0])
        self.conv1_1 = ConvBlock(dec_ch[1] + dec_ch[2], dec_ch[1])
        self.conv2_1 = ConvBlock(dec_ch[2] + dec_ch[3], dec_ch[2])
        self.conv3_1 = ConvBlock(dec_ch[3] + dec_ch[4], dec_ch[3])

        self.conv0_2 = ConvBlock(dec_ch[0] * 2 + dec_ch[1], dec_ch[0])
        self.conv1_2 = ConvBlock(dec_ch[1] * 2 + dec_ch[2], dec_ch[1])
        self.conv2_2 = ConvBlock(dec_ch[2] * 2 + dec_ch[3], dec_ch[2])

        self.conv0_3 = ConvBlock(dec_ch[0] * 3 + dec_ch[1], dec_ch[0])
        self.conv1_3 = ConvBlock(dec_ch[1] * 3 + dec_ch[2], dec_ch[1])

        self.conv0_4 = ConvBlock(dec_ch[0] * 4 + dec_ch[1], dec_ch[0])

        self.head = nn.Conv2d(dec_ch[0], num_classes, kernel_size=1)

    @staticmethod
    def _up(x, target):
        return F.interpolate(x, size=target.shape[-2:], mode="bilinear", align_corners=False)

    def forward(self, x):
        input_hw = x.shape[-2:]

        feats = self.encoder(x)
        x0_0 = self.cbam0(self.proj0(feats[0]))
        x1_0 = self.cbam1(self.proj1(feats[1]))
        x2_0 = self.cbam2(self.proj2(feats[2]))
        x3_0 = self.cbam3(self.proj3(feats[3]))
        x4_0 = self.cbam4(self.proj4(feats[4]))
        x4_0 = self.mhsa(x4_0)

        s0 = self.ag0(x0_0, x1_0)
        s1 = self.ag1(x1_0, x2_0)
        s2 = self.ag2(x2_0, x3_0)
        s3 = self.ag3(x3_0, x4_0)

        x0_1 = self.conv0_1(torch.cat([s0, self._up(x1_0, x0_0)], dim=1))
        x1_1 = self.conv1_1(torch.cat([s1, self._up(x2_0, x1_0)], dim=1))
        x2_1 = self.conv2_1(torch.cat([s2, self._up(x3_0, x2_0)], dim=1))
        x3_1 = self.conv3_1(torch.cat([s3, self._up(x4_0, x3_0)], dim=1))

        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self._up(x1_1, x0_0)], dim=1))
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self._up(x2_1, x1_0)], dim=1))
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self._up(x3_1, x2_0)], dim=1))

        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self._up(x1_2, x0_0)], dim=1))
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self._up(x2_2, x1_0)], dim=1))

        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self._up(x1_3, x0_0)], dim=1))

        logits = self.head(x0_4)
        logits = F.interpolate(logits, size=input_hw, mode="bilinear", align_corners=False)
        return logits


model = HighPerfFloodUNetPP(num_classes=1, pretrained=True).to(CONFIG["DEVICE"])
num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model params: {num_params:.2f}M")

with torch.no_grad():
    dummy = torch.randn(1, 3, CONFIG["IMAGE_HEIGHT"], CONFIG["IMAGE_WIDTH"]).to(CONFIG["DEVICE"])
    out = model(dummy)
print("Dummy output shape:", out.shape)

Model params: 9.10M
Dummy output shape: torch.Size([1, 1, 512, 512])


In [57]:
def dice_loss_with_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probs * targets).sum(dim=dims)
    union = probs.sum(dim=dims) + targets.sum(dim=dims)
    dice = (2.0 * intersection + eps) / (union + eps)
    return 1.0 - dice


def mask_boundary(mask):
    # Morphological gradient approximation
    max_pool = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
    min_pool = -F.max_pool2d(-mask, kernel_size=3, stride=1, padding=1)
    return (max_pool - min_pool).clamp(0.0, 1.0)


def combined_segmentation_loss(logits, targets, w_dice=1.0, w_bce=1.0, w_boundary=0.5):
    dice_term = dice_loss_with_logits(logits, targets).mean()
    bce_term = F.binary_cross_entropy_with_logits(logits, targets)

    probs = torch.sigmoid(logits)
    pred_boundary = mask_boundary(probs)
    true_boundary = mask_boundary(targets)
    boundary_term = F.binary_cross_entropy(pred_boundary, true_boundary)

    total = w_dice * dice_term + w_bce * bce_term + w_boundary * boundary_term
    parts = {
        "dice": float(dice_term.item()),
        "bce": float(bce_term.item()),
        "boundary": float(boundary_term.item()),
    }
    return total, parts


def iou_score_from_logits(logits, targets, threshold=0.5, eps=1e-8):
    preds = (torch.sigmoid(logits) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return float(iou.mean().item())


def precision_recall_f1_from_logits(logits, targets, threshold=0.5, eps=1e-8):
    preds = (torch.sigmoid(logits) > threshold).float()
    tp = (preds * targets).sum()
    fp = (preds * (1 - targets)).sum()
    fn = ((1 - preds) * targets).sum()

    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    return float(precision.item()), float(recall.item()), float(f1.item())


def rand_bbox(size, lam):
    _, _, h, w = size
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w = int(w * cut_ratio)
    cut_h = int(h * cut_ratio)

    cx = np.random.randint(w)
    cy = np.random.randint(h)

    x1 = np.clip(cx - cut_w // 2, 0, w)
    y1 = np.clip(cy - cut_h // 2, 0, h)
    x2 = np.clip(cx + cut_w // 2, 0, w)
    y2 = np.clip(cy + cut_h // 2, 0, h)
    return x1, y1, x2, y2


def apply_cutmix(images, masks, p=0.3, alpha=1.0):
    if np.random.rand() > p or images.size(0) < 2:
        return images, masks

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(images.size(0), device=images.device)
    x1, y1, x2, y2 = rand_bbox(images.size(), lam)

    images = images.clone()
    masks = masks.clone()
    images[:, :, y1:y2, x1:x2] = images[rand_index, :, y1:y2, x1:x2]
    masks[:, :, y1:y2, x1:x2] = masks[rand_index, :, y1:y2, x1:x2]
    return images, masks


def freeze_encoder(model, freeze=True):
    for p in model.encoder.parameters():
        p.requires_grad = not freeze


def build_optimizer(model, encoder_lr, decoder_lr, weight_decay):
    encoder_params = []
    decoder_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("encoder."):
            encoder_params.append(p)
        else:
            decoder_params.append(p)

    param_groups = [{"params": decoder_params, "lr": decoder_lr}]
    if len(encoder_params) > 0 and encoder_lr is not None:
        param_groups.append({"params": encoder_params, "lr": encoder_lr})

    optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)
    return optimizer


def build_warmup_cosine_scheduler(optimizer, total_epochs, warmup_epochs):
    warmup_epochs = min(warmup_epochs, max(total_epochs - 1, 1))

    def lr_lambda(epoch_idx):
        if epoch_idx < warmup_epochs:
            return float(epoch_idx + 1) / float(max(1, warmup_epochs))
        progress = (epoch_idx - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


@torch.no_grad()
def validate_one_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_iou = 0.0
    total_precision = 0.0
    total_recall = 0.0
    total_f1 = 0.0

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(images)
        loss, _ = combined_segmentation_loss(logits, masks)

        total_loss += float(loss.item())
        total_iou += iou_score_from_logits(logits, masks)
        p, r, f1 = precision_recall_f1_from_logits(logits, masks)
        total_precision += p
        total_recall += r
        total_f1 += f1

    n = max(len(loader), 1)
    return {
        "loss": total_loss / n,
        "iou": total_iou / n,
        "precision": total_precision / n,
        "recall": total_recall / n,
        "f1": total_f1 / n,
    }

In [58]:
def train_highperf(model, train_loader, val_loader, config):
    device = config["DEVICE"]
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    micro_batch = config.get("MICRO_BATCH_SIZE", config["BATCH_SIZE"])
    accum_steps = max(1, math.ceil(config["BATCH_SIZE"] / micro_batch))
    max_steps = config.get("MAX_STEPS_PER_EPOCH", None)

    freeze_epochs = min(config["FREEZE_EPOCHS"], config["EPOCHS"])
    best_val_iou = -1.0
    history = []

    # Phase 1: freeze encoder
    freeze_encoder(model, freeze=True)
    optimizer = build_optimizer(
        model,
        encoder_lr=None,
        decoder_lr=config["LR"],
        weight_decay=config["WEIGHT_DECAY"],
    )
    scheduler = build_warmup_cosine_scheduler(
        optimizer,
        total_epochs=max(freeze_epochs, 1),
        warmup_epochs=config["WARMUP_EPOCHS"],
    )

    print("Mulai training phase-1 (encoder frozen)")
    print(
        f"Gradient accumulation aktif: micro_batch={micro_batch}, "
        f"target_batch={config['BATCH_SIZE']}, accum_steps={accum_steps}"
    )

    for epoch in range(config["EPOCHS"]):
        if epoch == freeze_epochs and freeze_epochs < config["EPOCHS"]:
            # Phase 2: unfreeze encoder
            freeze_encoder(model, freeze=False)
            optimizer = build_optimizer(
                model,
                encoder_lr=config["LR_UNFREEZE"],
                decoder_lr=config["LR"],
                weight_decay=config["WEIGHT_DECAY"],
            )
            scheduler = build_warmup_cosine_scheduler(
                optimizer,
                total_epochs=max(config["EPOCHS"] - freeze_epochs, 1),
                warmup_epochs=min(config["WARMUP_EPOCHS"], max(config["EPOCHS"] - freeze_epochs - 1, 1)),
            )
            print("Masuk phase-2 (encoder unfrozen)")

        model.train()
        running_loss = 0.0
        running_iou = 0.0

        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['EPOCHS']}", leave=False)
        for step_idx, (images, masks) in enumerate(pbar):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            images, masks = apply_cutmix(
                images,
                masks,
                p=config["CUTMIX_PROB"],
                alpha=config["CUTMIX_ALPHA"],
            )

            with torch.autocast(device_type="cuda" if use_amp else "cpu", enabled=use_amp):
                logits = model(images)

            # Hitung loss di luar autocast agar BCE boundary aman
            loss, loss_parts = combined_segmentation_loss(logits.float(), masks.float())
            loss_for_backward = loss / accum_steps

            scaler.scale(loss_for_backward).backward()

            # Step optimizer setiap accum_steps (atau batch terakhir)
            is_update_step = ((step_idx + 1) % accum_steps == 0) or ((step_idx + 1) == len(train_loader))
            if is_update_step:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            batch_iou = iou_score_from_logits(logits.detach(), masks)
            running_loss += float(loss.item())
            running_iou += batch_iou

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                iou=f"{batch_iou:.4f}",
                d=f"{loss_parts['dice']:.4f}",
                b=f"{loss_parts['bce']:.4f}",
                edge=f"{loss_parts['boundary']:.4f}",
            )

            if max_steps is not None and (step_idx + 1) >= max_steps:
                # Pastikan grad terakhir tetap di-step jika break sebelum is_update_step
                if not is_update_step:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                break

        scheduler.step()

        denom = max(1, (step_idx + 1) if max_steps is not None else len(train_loader))
        train_loss = running_loss / denom
        train_iou = running_iou / denom
        val_metrics = validate_one_epoch(model, val_loader, device)

        current_lr = optimizer.param_groups[0]["lr"]
        log_line = (
            f"Epoch [{epoch+1}/{config['EPOCHS']}] | "
            f"LR: {current_lr:.2e} | "
            f"Train Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | Val IoU: {val_metrics['iou']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f}"
        )
        print(log_line)

        history.append(
            {
                "epoch": epoch + 1,
                "lr": current_lr,
                "train_loss": train_loss,
                "train_iou": train_iou,
                "val_loss": val_metrics["loss"],
                "val_iou": val_metrics["iou"],
                "val_precision": val_metrics["precision"],
                "val_recall": val_metrics["recall"],
                "val_f1": val_metrics["f1"],
            }
        )

        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": config,
                    "best_val_iou": best_val_iou,
                    "epoch": epoch + 1,
                },
                config["CHECKPOINT_PATH"],
            )
            print(f"Checkpoint terbaik disimpan (Val IoU={best_val_iou:.4f})")

    print("Training selesai.")
    return pd.DataFrame(history)

In [59]:
# Optional quick sanity run (1 epoch, beberapa step saja)
quick_cfg = dict(CONFIG)
quick_cfg["EPOCHS"] = 1
quick_cfg["FREEZE_EPOCHS"] = 0
quick_cfg["WARMUP_EPOCHS"] = 1
quick_cfg["MAX_STEPS_PER_EPOCH"] = 4
quick_cfg["CUTMIX_PROB"] = 0.0
history_quick = train_highperf(model, train_loader, val_loader, quick_cfg)
history_quick.tail()

Mulai training phase-1 (encoder frozen)
Gradient accumulation aktif: micro_batch=2, target_batch=16, accum_steps=8
Masuk phase-2 (encoder unfrozen)


Epoch 1/1:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch [1/1] | LR: 1.00e-04 | Train Loss: 1.1798 | Train IoU: 0.5692 | Val Loss: 1.3276 | Val IoU: 0.4314 | Val F1: 0.5976
Checkpoint terbaik disimpan (Val IoU=0.4314)
Training selesai.


,epoch,lr,train_loss,train_iou,val_loss,val_iou,val_precision,val_recall,val_f1
0,1,0.0001,1.179804,0.569218,1.32757,0.431351,0.440528,0.96695,0.597617


In [60]:
# Full training run (sesuai CONFIG)
history_df = train_highperf(model, train_loader, val_loader, CONFIG)
history_df.tail()

Mulai training phase-1 (encoder frozen)
Gradient accumulation aktif: micro_batch=2, target_batch=16, accum_steps=8


Epoch 1/150:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch [1/150] | LR: 4.00e-05 | Train Loss: 1.1933 | Train IoU: 0.5366 | Val Loss: 1.0739 | Val IoU: 0.6105 | Val F1: 0.7512
Checkpoint terbaik disimpan (Val IoU=0.6105)


Epoch 2/150:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch [2/150] | LR: 6.00e-05 | Train Loss: 1.0795 | Train IoU: 0.6090 | Val Loss: 0.9286 | Val IoU: 0.7005 | Val F1: 0.8209
Checkpoint terbaik disimpan (Val IoU=0.7005)


...

Epoch 149/150:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch [149/150] | LR: 1.87e-08 | Train Loss: 0.3783 | Train IoU: 0.8527 | Val Loss: 0.4586 | Val IoU: 0.8043 | Val F1: 0.8942


Epoch 150/150:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch [150/150] | LR: 0.00e+00 | Train Loss: 0.3862 | Train IoU: 0.8489 | Val Loss: 0.4437 | Val IoU: 0.8080 | Val F1: 0.8965
Training selesai.


,epoch,lr,train_loss,train_iou,val_loss,val_iou,val_precision,val_recall,val_f1
145,146,2.982166e-07,0.387936,0.851732,0.452475,0.806463,0.896146,0.900322,0.895290
146,147,1.678199e-07,0.385231,0.849000,0.448050,0.805990,0.906486,0.889700,0.895226
147,148,7.460983e-08,0.388060,0.846814,0.446635,0.808559,0.901862,0.896920,0.896503
148,149,1.865594e-08,0.378292,0.852678,0.458594,0.804287,0.895937,0.898106,0.894199
149,150,0.000000e+00,0.386177,0.848940,0.443681,0.808036,0.906507,0.892066,0.896545


In [ ]:
@torch.no_grad()
def tta_predict_proba(model, images):
    # 4-way TTA: identity, HFlip, VFlip, HVFlip
    variants = [
        (lambda x: x, lambda y: y),
        (lambda x: torch.flip(x, dims=[3]), lambda y: torch.flip(y, dims=[3])),
        (lambda x: torch.flip(x, dims=[2]), lambda y: torch.flip(y, dims=[2])),
        (lambda x: torch.flip(x, dims=[2, 3]), lambda y: torch.flip(y, dims=[2, 3])),
    ]

    probs = []
    for aug, deaug in variants:
        logits = model(aug(images))
        probs.append(torch.sigmoid(deaug(logits)))
    return torch.mean(torch.stack(probs, dim=0), dim=0)


def dense_crf_refine(image_rgb_uint8, prob_map, n_iters=5):
    # prob_map shape: [H, W], values [0,1]
    if not HAS_DCRF:
        return prob_map

    h, w = prob_map.shape
    unary = np.stack([1.0 - prob_map, prob_map], axis=0)
    unary = unary_from_softmax(unary)

    d = dcrf.DenseCRF2D(w, h, 2)
    d.setUnaryEnergy(unary)
    d.addPairwiseGaussian(sxy=3, compat=3)
    d.addPairwiseBilateral(sxy=50, srgb=10, rgbim=image_rgb_uint8, compat=5)

    q = d.inference(n_iters)
    q = np.array(q).reshape((2, h, w))
    return q[1]


@torch.no_grad()
def evaluate_tta_crf(model, loader, device):
    model.eval()
    total_iou = 0.0
    total_precision = 0.0
    total_recall = 0.0
    total_f1 = 0.0
    count = 0

    for images, masks in tqdm(loader, desc="Eval TTA+CRF", leave=False):
        images = images.to(device)
        masks = masks.to(device)

        probs = tta_predict_proba(model, images)

        preds_list = []
        for i in range(probs.size(0)):
            prob = probs[i, 0].detach().cpu().numpy()
            img_np = images[i].detach().cpu().numpy().transpose(1, 2, 0)
            img_np = np.clip(
                img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]),
                0,
                1,
            )
            img_uint8 = (img_np * 255).astype(np.uint8)
            prob_refined = dense_crf_refine(img_uint8, prob)
            preds_list.append((prob_refined > 0.5).astype(np.float32))

        preds = torch.from_numpy(np.stack(preds_list)).unsqueeze(1).to(device)

        tp = (preds * masks).sum().item()
        fp = (preds * (1 - masks)).sum().item()
        fn = ((1 - preds) * masks).sum().item()
        union = preds.sum().item() + masks.sum().item() - tp

        iou = tp / (union + 1e-8)
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        total_iou += iou
        total_precision += precision
        total_recall += recall
        total_f1 += f1
        count += 1

    count = max(count, 1)
    return {
        "iou": total_iou / count,
        "precision": total_precision / count,
        "recall": total_recall / count,
        "f1": total_f1 / count,
    }


# Load checkpoint terbaik sebelum evaluasi akhir
if os.path.exists(CONFIG["CHECKPOINT_PATH"]):
    ckpt = torch.load(CONFIG["CHECKPOINT_PATH"], map_location=CONFIG["DEVICE"])
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Checkpoint loaded dari epoch {ckpt['epoch']} | best IoU={ckpt['best_val_iou']:.4f}")
else:
    print("Checkpoint belum ditemukan, evaluasi pakai bobot saat ini.")

val_plain = validate_one_epoch(model, val_loader, CONFIG["DEVICE"])
val_tta_crf = evaluate_tta_crf(model, val_loader, CONFIG["DEVICE"])

print("\n--- Validation (Plain) ---")
print(val_plain)
print("\n--- Validation (TTA + DenseCRF) ---")
print(val_tta_crf)

In [ ]:
@torch.no_grad()
def show_predictions(model, loader, device, num_samples=3):
    model.eval()
    images, masks = next(iter(loader))
    images = images.to(device)
    masks = masks.to(device)

    probs = tta_predict_proba(model, images)
    preds = (probs > 0.5).float()

    n = min(num_samples, images.size(0))
    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(n):
        img_np = images[i].detach().cpu().numpy().transpose(1, 2, 0)
        img_np = np.clip(
            img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]),
            0,
            1,
        )
        gt = masks[i, 0].detach().cpu().numpy()
        pr = preds[i, 0].detach().cpu().numpy()

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title("Image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(gt, cmap="gray")
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(img_np)
        axes[i, 2].imshow(pr, cmap="jet", alpha=0.4)
        axes[i, 2].set_title("Prediction")
        axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()


show_predictions(model, val_loader, CONFIG["DEVICE"], num_samples=3)